In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\emotion_detection\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\emotion_detection'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [7]:
#entity 1st update

@dataclass(frozen= True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_name: Path

In [8]:
from emotion_detection.constant import *
from emotion_detection.utils.common import *

In [9]:
#2nd configyration manager update

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            test_data_path=Path(config.test_data_path),
            model_path=Path(config.model_path),
            metric_file_name=Path(config.metric_file_name)
        )

        return model_evaluation_config

In [10]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import json

In [11]:
#3rd component update

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def evaluate(self):
        # 1. Image Transforms
        val_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        # 2. Load Test Dataset
        test_dataset = datasets.ImageFolder(root=self.config.test_data_path, transform=val_transform)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

        # 3. Reconstruct Model Architecture & Load Trained Weights
        resnet = models.resnet18(weights=None)
        resnet.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(resnet.fc.in_features, len(test_dataset.classes))
        )
        model = resnet.to(self.device)
        model.load_state_dict(torch.load(self.config.model_path, map_location=self.device))
        model.eval()

        # 4. Evaluation Loop
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(self.device)
                outputs = model(images)
                preds = outputs.argmax(1).cpu().numpy()

                all_preds.extend(preds)
                all_labels.extend(labels.numpy())

        # 5. Compute Metrics
        accuracy = accuracy_score(all_labels, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

        print("--- Classification Report ---")
        print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))

        # 6. Save Metrics to JSON
        scores = {
            "test_accuracy": float(accuracy),
            "weighted_precision": float(precision),
            "weighted_recall": float(recall),
            "weighted_f1_score": float(f1)
        }
        
        with open(self.config.metric_file_name, "w") as f:
            json.dump(scores, f, indent=4)

In [13]:
try:
    config_manager = configurationManager()
    model_evaluation_config = config_manager.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.evaluate()
except Exception as e:
    raise e

[2026-07-21 12:09:26,946: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-07-21 12:09:26,951: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-07-21 12:09:26,953: INFO: common: created directory at artifacts]
[2026-07-21 12:09:26,956: INFO: common: created directory at artifacts/model_evaluation]
--- Classification Report ---
              precision    recall  f1-score   support

       angry       0.90      0.86      0.88      1185
     disgust       0.47      0.41      0.44       160
        fear       0.57      0.47      0.52        74
       happy       0.70      0.78      0.74       680
     neutral       0.67      0.70      0.68       162
         sad       0.72      0.72      0.72       478
    surprise       0.78      0.75      0.77       329

    accuracy                           0.77      3068
   macro avg       0.69      0.67      0.68      3068
weighted avg       0.77      0.77      0.77      3068

